# SatQuery on Google Colab

This notebook installs and runs the real Engine, Platform API, and Vite frontend in the Colab VM. Engine Agent chat uses the repository's OpenAI-compatible streaming adapter and a real OpenAI API key; Engine investigation smoke tests use the configured live Open-Meteo connector. It never substitutes canned model or provider responses.

**Limits:** Colab storage is ephemeral unless you export stopped-process database snapshots to Drive. The notebook does not expose a production service, add authentication, or validate provider licensing. A provider/model failure is reported as a failed smoke test, not success.

## 1. Runtime and repository

Select a Python 3.12 Colab runtime. GPU is not required for Platform/Engine operations or hosted Agent inference. Node 22 is required for the Vite frontend. The `vision` extra is intentionally omitted: hosted Agent vision uses the selected remote model, while local BLIP/CLIP inference is a separate, large optional installation.

In [ ]:
import importlib, os, shutil, subprocess, sys
from pathlib import Path

if sys.version_info < (3, 12):
    raise RuntimeError(f"Python 3.12+ required by the Platform; runtime has {sys.version.split()[0]}. Select a compatible Colab runtime.")
ROOT = Path('/content/SatQuery-AI')
if not (ROOT / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/NotAzie/SatQuery-AI.git', str(ROOT)], check=True)
cpu_count = os.cpu_count() or 1
memory_total_gib = None
if Path('/proc/meminfo').exists():
    for line in Path('/proc/meminfo').read_text().splitlines():
        if line.startswith('MemTotal:'):
            memory_total_gib = round(int(line.split()[1]) / 1024**2, 1)
            break
disk = shutil.disk_usage('/content')
gpu_result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
gpus = [line.strip() for line in gpu_result.stdout.splitlines() if line.strip()] if gpu_result.returncode == 0 else []
revision = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=ROOT, capture_output=True, text=True, check=True).stdout.strip()
print(f'Python: {sys.version.split()[0]} | CPUs: {cpu_count} | RAM: {memory_total_gib if memory_total_gib is not None else "unknown"} GiB')
print('GPU:', '; '.join(gpus) if gpus else 'not detected (CPU execution available)')
print(f'Free /content storage: {disk.free / 1024**3:.1f} GiB')
print('Repository revision:', revision)
print('Repository:', ROOT)

## 2. Install exact project requirements

The Engine and Platform use separate Python environments. `geospatial` and `raster` support the documented spatial/raster workflows; neither installs local vision models. npm uses the committed lockfile. Node 22 is installed only if the Colab image does not already provide it.

In [ ]:
node_major = subprocess.run(['bash', '-lc', 'node -p "process.versions.node.split(\".\")[0]"'], text=True, capture_output=True)
if node_major.returncode != 0 or node_major.stdout.strip() != '22':
    subprocess.run(['bash', '-lc', 'apt-get update && apt-get install -y ca-certificates curl gnupg && curl -fsSL https://deb.nodesource.com/setup_22.x | bash - && apt-get install -y nodejs'], check=True)
subprocess.run(['node', '--version'], check=True)
engine_dir = ROOT / 'engine' / 'satquery-ai'
app_dir = ROOT / 'APP'
engine_python = engine_dir / '.venv' / 'bin' / 'python'
app_python = app_dir / '.venv' / 'bin' / 'python'
if not engine_python.exists():
    subprocess.run([sys.executable, '-m', 'venv', str(engine_dir / '.venv')], check=True)
if not app_python.exists():
    subprocess.run([sys.executable, '-m', 'venv', str(app_dir / '.venv')], check=True)
subprocess.run([str(engine_python), '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([str(engine_python), '-m', 'pip', 'install', '-e', '.[geospatial,raster]'], cwd=engine_dir, check=True)
subprocess.run([str(app_python), '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([str(app_python), '-m', 'pip', 'install', '-e', 'backend'], cwd=app_dir, check=True)
subprocess.run(['npm', 'ci'], cwd=app_dir / 'frontend', check=True)
print('Engine, Platform API, and frontend dependencies installed.')

## 3. Colab Secrets and provider configuration

For hosted Agent chat, add `OPENAI_API_KEY` to Colab **Secrets**, grant this notebook access, and set `AGENT_MODEL` to an OpenAI Chat Completions model available to your account. Both are optional if you only want the Platform and Engine-backed data workflow. Optionally add `VITE_CESIUM_ION_TOKEN`. Do not put credentials in notebook source or print them. Set `AGENT_SUPPORTS_VISION=True` only if the selected model accepts image input; this is an operator assertion, not a model capability probe.

In [ ]:
try:
    userdata = importlib.import_module('google.colab.userdata')
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    CESIUM_ION_TOKEN = userdata.get('VITE_CESIUM_ION_TOKEN')
except ImportError as exc:
    raise RuntimeError('Run this notebook in Google Colab and configure Colab Secrets.') from exc
AGENT_MODEL = ''  # Set an OpenAI Chat Completions model enabled for your account.
AGENT_SUPPORTS_VISION = False  # Set True only when the chosen model supports image input.
agent_configured = bool(OPENAI_API_KEY and AGENT_MODEL.strip())
os.environ.pop('OPENAI_API_KEY', None)  # Keep Agent credentials separate from the legacy planner.
if agent_configured:
    os.environ['SATQUERY_AGENT_OPENAI_API_KEY'] = OPENAI_API_KEY
    os.environ['SATQUERY_AGENT_OPENAI_MODEL'] = AGENT_MODEL.strip()
    os.environ['SATQUERY_AGENT_OPENAI_SUPPORTS_VISION'] = str(AGENT_SUPPORTS_VISION).lower()
else:
    os.environ.pop('SATQUERY_AGENT_OPENAI_API_KEY', None)
    os.environ.pop('SATQUERY_AGENT_OPENAI_MODEL', None)
    os.environ.pop('SATQUERY_AGENT_OPENAI_SUPPORTS_VISION', None)
    print('Agent chat disabled: configure the OpenAI key and AGENT_MODEL to enable its smoke test.')
if CESIUM_ION_TOKEN:
    os.environ['VITE_CESIUM_ION_TOKEN'] = CESIUM_ION_TOKEN
os.environ['SATQUERY_CONNECTORS_CONFIG'] = str(ROOT / 'engine' / 'configs' / 'connectors.example.json')
if agent_configured: print('OpenAI Agent key loaded from Colab Secrets; value not displayed.')
if agent_configured: print('Agent model configured:', AGENT_MODEL.strip())
print('Example Engine connector manifest configured.')

## 4. Local persistence and optional Google Drive snapshots

SQLite must stay on the Colab VM while services are running. Optionally mount Drive and restore a **stopped-run** snapshot before startup; after stopping services, this notebook copies databases and image files to Drive. A reset loses unsaved `/content` data. Drive snapshots are not live replication or a backup service.

In [ ]:
import shutil
DATA_DIR = Path('/content/satquery-data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
ENGINE_DB = DATA_DIR / 'engine.sqlite'
PLATFORM_DB = app_dir / 'var' / 'platform.sqlite3'
PLATFORM_IMAGES = app_dir / 'var' / 'images'
USE_GOOGLE_DRIVE = False  # Set True to mount Drive for stopped-run snapshots.
DRIVE_SNAPSHOT_DIR = Path('/content/drive/MyDrive/SatQuery/snapshots')
if USE_GOOGLE_DRIVE:
    drive = importlib.import_module('google.colab.drive')
    drive.mount('/content/drive')
    snapshots = sorted(path for path in DRIVE_SNAPSHOT_DIR.iterdir() if path.is_dir()) if DRIVE_SNAPSHOT_DIR.exists() else []
    latest = snapshots[-1] if snapshots else None
    if latest is not None:
        if (latest / 'engine.sqlite').exists(): shutil.copy2(latest / 'engine.sqlite', ENGINE_DB)
        if (latest / 'platform.sqlite3').exists():
            PLATFORM_DB.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(latest / 'platform.sqlite3', PLATFORM_DB)
        if (latest / 'images').exists(): shutil.copytree(latest / 'images', PLATFORM_IMAGES, dirs_exist_ok=True)
engine_config = engine_dir / 'colab-platform.json'
engine_config.write_text(__import__('json').dumps({'storage': {'earth_memory_path': str(ENGINE_DB)}}, indent=2))
print('Engine SQLite:', ENGINE_DB)
print('Platform SQLite:', PLATFORM_DB)
print('Drive snapshots enabled:', USE_GOOGLE_DRIVE)

## 5. Start Engine, Platform API, and frontend

The Engine registers `openai/<model>` when the model/key environment is present. The Platform uses that exact ID. Both APIs bind inside the VM; Vite is exposed through Colab's port proxy.

In [ ]:
import time, urllib.request

engine_env = os.environ.copy()
engine_env.update({'SATQUERY_HOST': '127.0.0.1', 'SATQUERY_PORT': '9000', 'SATQUERY_PLATFORM_CONFIG': str(engine_config)})
platform_env = os.environ.copy()
platform_env.update({'EARTH_ENGINE__KIND': 'http', 'EARTH_ENGINE__BASE_URL': 'http://127.0.0.1:9000', 'EARTH_SERVER__HOST': '127.0.0.1'})
if agent_configured: platform_env['EARTH_AGENT__MODEL_ID'] = f'openai/{AGENT_MODEL.strip()}'
frontend_env = os.environ.copy()
frontend_env['VITE_API_PROXY_TARGET'] = 'http://127.0.0.1:8000'
log_dir = Path('/content/satquery-logs')
log_dir.mkdir(exist_ok=True)
logs = {name: open(log_dir / f'{name}.log', 'w') for name in ('engine', 'platform', 'frontend')}
processes = {}
processes['engine'] = subprocess.Popen([str(engine_python), '-m', 'satquery', 'serve', '--host', '127.0.0.1', '--port', '9000'], cwd=engine_dir, env=engine_env, stdout=logs['engine'], stderr=subprocess.STDOUT, start_new_session=True)
processes['platform'] = subprocess.Popen([str(app_python), '-m', 'uvicorn', 'earth_platform.main:create_app', '--factory', '--app-dir', 'backend/src', '--host', '127.0.0.1', '--port', '8000'], cwd=app_dir, env=platform_env, stdout=logs['platform'], stderr=subprocess.STDOUT, start_new_session=True)
processes['frontend'] = subprocess.Popen(['npm', 'run', 'dev', '--', '--host', '0.0.0.0'], cwd=app_dir / 'frontend', env=frontend_env, stdout=logs['frontend'], stderr=subprocess.STDOUT, start_new_session=True)
def wait_http(url, process, label, timeout=180):
    deadline = time.time() + timeout
    last_error = None
    while time.time() < deadline:
        if process.poll() is not None: raise RuntimeError(f'{label} exited; inspect {log_dir / (label.lower()+".log")}')
        try:
            with urllib.request.urlopen(url, timeout=3) as response: return response.status
        except Exception as exc:
            last_error = exc
            time.sleep(1)
    raise TimeoutError(f'{label} did not respond at {url}: {last_error}')
print('Engine health HTTP:', wait_http('http://127.0.0.1:9000/api/v1/health', processes['engine'], 'engine'))
print('Platform liveness HTTP:', wait_http('http://127.0.0.1:8000/healthz', processes['platform'], 'platform'))
print('Frontend HTTP:', wait_http('http://127.0.0.1:5173', processes['frontend'], 'frontend'))
output = importlib.import_module('google.colab.output')
output.serve_kernel_port_as_window(5173)

## 6. Verify health and a real Agent conversation

Engine health and Platform liveness are not proof that the model works. When credentials and a model are configured, this cell creates a real Engine Agent session, sends a request to the hosted model, parses actual SSE events, and raises an error unless a non-empty assistant response completes. Without them, the Agent check is explicitly skipped; API/provider errors are surfaced.

In [ ]:
import json, urllib.request

def api_json(url, payload=None):
    data = None if payload is None else json.dumps(payload).encode()
    request = urllib.request.Request(url, data=data, headers={'Content-Type': 'application/json'})
    with urllib.request.urlopen(request, timeout=30) as response: return response.status, json.loads(response.read())
engine_status, engine_health = api_json('http://127.0.0.1:9000/api/v1/health')
platform_status, platform_ready = api_json('http://127.0.0.1:8000/readyz')
_, discovery = api_json('http://127.0.0.1:8000/api/v1/intelligence/engine-discovery')
print('Engine health:', engine_status, engine_health)
print('Platform readiness:', platform_status, platform_ready)
if not agent_configured:
    print('Agent conversation smoke test skipped: configure the OpenAI key and AGENT_MODEL.')
else:
    _, session = api_json('http://127.0.0.1:9000/api/v1/agent/sessions', {})
    session_id = session['session_id']
    turn = urllib.request.Request(f'http://127.0.0.1:9000/api/v1/agent/sessions/{session_id}/turns', data=json.dumps({'message': 'Introduce SatQuery and state which tools you can use. Do not claim to have acquired data.', 'model_id': f'openai/{AGENT_MODEL.strip()}', 'execution_authorized': False}).encode(), headers={'Content-Type': 'application/json'})
    events = []
    with urllib.request.urlopen(turn, timeout=300) as response:
        for raw_line in response:
            line = raw_line.decode('utf-8').strip()
            if line.startswith('data: '): events.append(json.loads(line[6:]))
    text = ''.join(event.get('payload', {}).get('text', '') for event in events if event['kind'] == 'assistant.delta')
    completed = any(event['kind'] == 'turn.completed' for event in events)
    failures = [event for event in events if event['kind'] in {'agent.error', 'tool.failed'}]
    if failures: raise RuntimeError(f'Real Agent smoke test returned errors: {failures}')
    if not completed or not text.strip(): raise RuntimeError(f'No completed non-empty Agent response. Events: {events}')
    print('Real Agent response:', text)
    print('Agent SSE events:', len(events), '| completed:', completed)

## 7. Execute one real Engine-backed investigation

This uses the example Open-Meteo weather connector and the repository's deterministic planner/executor. It requests a small point/time window. Upstream service, data coverage, or quota failures stop the cell; a registered connector alone is not counted as success.

In [ ]:
from datetime import datetime, timezone

query = {
    'point': [-122.3321, 47.6062],
    'temporal_start': datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0).isoformat(),
    'temporal_end': datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0).isoformat(),
    'requested_dataset_ids': ['weather-forecast'],
    'requested_variables': ['air_temperature_2m', 'precipitation_amount']
}
_, mission = api_json('http://127.0.0.1:9000/api/v1/platform/missions', {'name': 'Colab live provider smoke test', 'objective': 'Fetch Open-Meteo forecast data for the configured point.'})
mission_id = mission['mission_id']
_, created = api_json(f'http://127.0.0.1:9000/api/v1/platform/missions/{mission_id}/investigations', query)
investigation_id = created['investigation_id']
execution_status, executed = api_json(f'http://127.0.0.1:9000/api/v1/platform/investigations/{investigation_id}/execute', {})
_, observations = api_json(f'http://127.0.0.1:9000/api/v1/platform/investigations/{investigation_id}/observations')
if executed['status'] != 'succeeded': raise RuntimeError(f'Investigation did not succeed: {executed}')
if not observations: raise RuntimeError(f'Successful investigation returned no provider observations: {executed}')
print('Investigation ID:', investigation_id)
print('Execution HTTP:', execution_status)
print('Provider observations:', len(observations))
print('First observation metadata:', json.dumps(observations[0], indent=2)[:2500])

## 8. Stop services and optionally save Drive snapshot

Stop all processes before copying SQLite files so the snapshot is consistent. Drive is optional. Files written under `/content` disappear when the runtime is reclaimed.

In [ ]:
import signal

for name, process in processes.items():
    try:
        killpg = getattr(os, 'killpg', None)
        if killpg is None: process.terminate()
        else: killpg(process.pid, signal.SIGTERM)
    except ProcessLookupError: pass
for name, process in processes.items():
    try: process.wait(timeout=15)
    except subprocess.TimeoutExpired:
        try:
            killpg = getattr(os, 'killpg', None)
            if killpg is None: process.kill()
            else: killpg(process.pid, getattr(signal, 'SIGKILL', signal.SIGTERM))
        except ProcessLookupError: pass
        process.wait(timeout=10)
for handle in logs.values(): handle.close()
print('Engine, Platform API, and frontend stopped.')
if USE_GOOGLE_DRIVE:
    import shutil
    snapshot = DRIVE_SNAPSHOT_DIR / time.strftime('%Y%m%d-%H%M%S')
    snapshot.mkdir(parents=True, exist_ok=True)
    if ENGINE_DB.exists(): shutil.copy2(ENGINE_DB, snapshot / 'engine.sqlite')
    if PLATFORM_DB.exists(): shutil.copy2(PLATFORM_DB, snapshot / 'platform.sqlite3')
    if PLATFORM_IMAGES.exists(): shutil.copytree(PLATFORM_IMAGES, snapshot / 'images', dirs_exist_ok=True)
    print('Stopped-run snapshot saved to:', snapshot)
else:
    print('Drive snapshot skipped; /content data is ephemeral.')